# 第 59 课｜可验证奖励与策略梯度
## REINFORCE、Baseline 与 KL 约束

上一课从固定偏好对做 DPO。本课让当前策略**自己采样完整回答**，根据可验证奖励更新 Q/V LoRA。

**目标：** 能写出并检查 `采样 → 奖励 → 重算 log 概率 → baseline → 反向 → 更新`，区分任务奖励、代理损失、KL 估计和实际生成表现。

本课包含两种实际实验：有限动作分布上的**精确枚举**，以及从随机参数预热后进行的**小型字符 Transformer 在线采样训练**。都不是预训练大模型、真实数学推理能力或 GPU 性能评测。

使用已有 PyTorch 环境。Notebook 内嵌完整代码，不依赖同名脚本或任何前课文件。

## 0. 运行与文件约定

从全新 Python 3 内核依次运行。完整实验会实际训练，所有默认设置都在下面的 `TrainConfig` 中。独立脚本支持：

```bash
python lesson_59_reinforce_baselines_and_kl.py --checks-only
python lesson_59_reinforce_baselines_and_kl.py --math-only
python lesson_59_reinforce_baselines_and_kl.py
python lesson_59_reinforce_baselines_and_kl.py --mode loo_kl
```

本课只用 PyTorch 和标准库，不需要 Transformers、TRL、PEFT、联网语料或新模型。默认 CPU，显式请求 CUDA 不可用时会报错。

每次完整运行创建新的时间戳目录，保留所有训练采样、评分与模型。文件不是精确断点续训状态；不含优化器、数据游标和完整随机状态。不要加载来源不明的检查点。

**练习开关默认关闭。全部单元运行成功，不代表完成了独立练习。**

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第59课：可验证奖励与策略梯度——REINFORCE、Baseline与KL约束。

Python 3.10+，PyTorch。完全离线；真实训练小型字符Transformer的Q/V LoRA。
先用有限分布枚举核对梯度，再进行每批只更新一次的on-policy训练。
不是预训练LLM、TRL/PPO/GRPO复现，不是3060性能测试。
"""
from __future__ import annotations

# %% 01 公共约定
import argparse
import copy
import hashlib
import itertools
import json
import math
import platform
import random
import tempfile
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("需要已有PyTorch环境；请选择同一环境的Jupyter内核。") from exc

VERSION = "lesson59-v1"
PAD, BOS, EOS, IGNORE = 0, 1, 2, -100
SOURCES = {
    "policy_gradient": "https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html",
    "rloo_paper": "https://arxiv.org/html/2402.14740v1",
    "rloo_docs": "https://huggingface.co/docs/trl/rloo_trainer",
    "distributions": "https://docs.pytorch.org/docs/2.10/distributions.html",
    "multinomial": "https://docs.pytorch.org/docs/2.10/generated/torch.multinomial.html",
    "reproducibility": "https://docs.pytorch.org/docs/2.10/notes/randomness.html",
}

class LessonError(ValueError):
    """不满足本课约定时明确拒绝，不替换目标或伪造奖励。"""


def require(ok: bool, message: str) -> None:
    if not ok:
        raise LessonError(message)


def canonical(value) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def digest(value) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()


def write_json(path: Path, value) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True,
                              indent=2, allow_nan=False) + "\n", encoding="utf-8")


def tensor_hash(values: dict[str, torch.Tensor]) -> str:
    h = hashlib.sha256()
    for name, tensor in sorted(values.items()):
        t = tensor.detach().cpu().contiguous()
        h.update(canonical([name, list(t.shape), str(t.dtype)]).encode())
        h.update(bytes(t.reshape(-1).view(torch.uint8).tolist()))
    return h.hexdigest()


def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_device(name: str) -> torch.device:
    require(name in {"cpu", "cuda", "auto"}, "device只支持cpu/cuda/auto")
    if name == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    require(name != "cuda" or torch.cuda.is_available(), "CUDA不可用，不会静默回退。")
    return torch.device(name)


@dataclass(frozen=True)
class ModelConfig:
    vocab_size: int = 20
    dim: int = 32
    heads: int = 4
    layers: int = 2
    ff_dim: int = 64
    max_len: int = 24

    def validate(self) -> None:
        require(all(type(x) is int and x > 0 for x in asdict(self).values()), "配置必须为正整数")
        require(self.dim % self.heads == 0, "dim需要整除heads")
        require(self.vocab_size > 3 and self.max_len >= 16, "词表/上下文过小")


@dataclass(frozen=True)
class TrainConfig:
    seed: int = 59
    sft_steps: int = 40
    sft_lr: float = 0.003
    updates: int = 60
    prompts_per_update: int = 8
    group_size: int = 4
    max_new_tokens: int = 3
    rl_lr: float = 0.002
    kl_beta: float = 0.2
    rank: int = 4
    alpha: float = 8.0
    max_grad_norm: float = 1.0
    eval_samples: int = 8

    def validate(self) -> None:
        for name in ("seed", "sft_steps", "updates", "prompts_per_update", "group_size",
                     "max_new_tokens", "rank", "eval_samples"):
            value = getattr(self, name)
            require(type(value) is int and value > ( -1 if name == "seed" else 0), name+"非法")
        require(self.group_size >= 2, "LOO需要至少2个样本")
        require(self.max_new_tokens == 3, "本课固定三步回答视野，不在实验中更改任务")
        for name in ("sft_lr", "rl_lr", "alpha", "max_grad_norm"):
            v = getattr(self, name)
            require(type(v) in (int,float) and math.isfinite(v) and v > 0, name+"非法")
        require(type(self.kl_beta) in (int,float) and math.isfinite(self.kl_beta)
                and self.kl_beta >= 0, "kl_beta必须非负有限")

In [2]:
torch.set_num_threads(1)
DEVICE = "cpu"
OUTPUT_ROOT = Path("lesson_59_outputs")
seed_all(59)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("本课设备:", DEVICE, "CUDA可用:", torch.cuda.is_available())
print("默认训练配置:", asdict(TrainConfig()))

Python: 3.13.5
PyTorch: 2.10.0+cpu
本课设备: cpu CUDA可用: False
默认训练配置: {'seed': 59, 'sft_steps': 40, 'sft_lr': 0.003, 'updates': 60, 'prompts_per_update': 8, 'group_size': 4, 'max_new_tokens': 3, 'rl_lr': 0.002, 'kl_beta': 0.2, 'rank': 4, 'alpha': 8.0, 'max_grad_norm': 1.0, 'eval_samples': 8}


## 1. 从偏好对，到当前策略产生的轨迹

DPO 使用已给定的 chosen/rejected。这里每次更新前，用当前策略对同一问题独立采样 G 次，奖励器再检查这些**实际产生的回答**。不会把参考答案替换进生成结果。

本课任务是复制末尾两位十进制串，例如：

```text
提示：copy:03,12=
合格回答：12 + EOS
```

两个数均取 00–15，共 256 个不同提示，固定分为 160 训练、48 开发、48 测试。同语法、同数值范围的新组合，不是自然语言泛化。

为了专注策略梯度，回答策略的动作集合固定为 **十个数字＋EOS**。每一步在这个集合上重新归一化，温度为 1，不用 Top-k/Top-p；策略与参考必须采用同一集合。以下公式中的 π 都指这份**受限回答策略**，不是完整词表未经处理的 softmax。

最大新增长度固定为 3。遇到 EOS 或达到上限便结束。精确输出两个目标数字并产生 EOS 奖 1，否则奖 0。这里的 cap 是预先定义的有限任务终止，不是下载失败、运行异常等基础设施故障；后者直接中止更新，不伪造 0 奖励。

In [3]:
# %% 02 固定任务与精确奖励
class CharCodec:
    """固定复制任务的字符表。回答策略只允许十个数字及EOS。"""
    def __init__(self):
        self.tokens = ["<pad>", "<bos>", "<eos>"] + list("copy:0123456789,=")
        require(len(set(self.tokens)) == len(self.tokens), "词表不能重复")
        self.to_id = {s: i for i,s in enumerate(self.tokens)}
        self.answer_ids = [self.to_id[str(i)] for i in range(10)] + [EOS]

    def encode(self, text: str) -> list[int]:
        require(type(text) is str and all(c in self.to_id for c in text), "未知字符")
        return [self.to_id[c] for c in text]

    def decode_answer(self, ids: list[int]) -> str:
        require(all(type(i) is int and i in self.answer_ids and i != EOS for i in ids),
                "正文必须是数字token；EOS应在外层处理")
        return "".join(self.tokens[i] for i in ids)

    def contract(self) -> dict:
        return {"tokens": self.tokens, "answer_ids": self.answer_ids,
                "sampling": "categorical_with_replacement_temperature_1_no_topk_no_topp",
                "horizon": 3, "end": "first_EOS_or_cap", "version": VERSION}


def make_splits(seed: int = 59) -> dict[str, list[dict]]:
    """复制末尾两位十进制串；共享语法/数值范围，不共享完整prompt。"""
    rows = [{"id": f"copy_{a:02d}_{b:02d}", "prompt": f"copy:{a:02d},{b:02d}=",
             "answer": f"{b:02d}", "source": f"{a:02d},{b:02d}"}
            for a in range(16) for b in range(16)]
    random.Random(seed).shuffle(rows)
    out = {"train": rows[:160], "dev": rows[160:208], "test": rows[208:]}
    for name, group in out.items():
        for row in group:
            row["split"] = name
    return out


def validate_splits(splits: dict) -> None:
    require(set(splits) == {"train","dev","test"}, "划分名称错误")
    seen = set()
    for name,rows in splits.items():
        require(bool(rows), "划分不能为空")
        for row in rows:
            require(row["split"] == name, "划分标签错误")
            require(row["prompt"] not in seen, "prompt跨侧或重复")
            require(row["answer"] == row["prompt"][-3:-1], "任务标签与显式规则冲突")
            seen.add(row["prompt"])


def score_completion(text: str, stop_reason: str, expected: str) -> dict:
    """完成EOS且精确匹配两位串才奖1；cap/格式/错误答案均按本任务定义奖0。
    推理异常、数据错误不在此伪造为0奖励：它们抛出异常并中止更新。
    """
    require(type(text) is str and type(expected) is str, "评分字段必须是字符串")
    require(len(expected)==2 and expected.isascii() and expected.isdigit(), "参考答案必须两位数字")
    require(stop_reason in {"eos", "max_tokens"}, "执行故障不是低奖励样本")
    if stop_reason != "eos":
        return {"reward": 0.0, "reason": "missing_eos"}
    if len(text) != 2 or not text.isascii() or not text.isdigit():
        return {"reward": 0.0, "reason": "invalid_format"}
    return {"reward": float(text == expected), "reason": "correct" if text==expected else "wrong"}


def collate_completions(rows: list[dict], answers: list[list[int]], codec: CharCodec,
                        max_len: int, device="cpu") -> dict:
    require(bool(rows) and len(rows)==len(answers), "批次与回答不匹配")
    seqs, targets = [], []
    for row, ans in zip(rows, answers):
        require(bool(ans) and len(ans) <= 3, "回答长度需要1..3")
        require(all(type(t) is int and t in codec.answer_ids for t in ans), "回答token非法")
        require(EOS not in ans[:-1], "EOS后不能再有token")
        prefix = [BOS] + codec.encode(row["prompt"])
        require(len(prefix)+len(ans)<=max_len, "超出上下文；不静默截断")
        seqs.append(prefix+list(ans))
        targets.append([IGNORE]*len(prefix)+list(ans))
    width = max(map(len, seqs))
    ids = torch.tensor([s+[PAD]*(width-len(s)) for s in seqs], dtype=torch.long, device=device)
    labels = torch.tensor([s+[IGNORE]*(width-len(s)) for s in targets], dtype=torch.long, device=device)
    return {"input_ids":ids, "labels":labels}

In [4]:
codec = CharCodec()
splits = make_splits()
validate_splits(splits)
print("数据划分:", {k: len(v) for k,v in splits.items()})
print("词表:", codec.tokens)
print("回答动作:", [codec.tokens[i] for i in codec.answer_ids])
for text, stop in [("02", "eos"), ("2", "eos"), ("03", "eos"), ("02", "max_tokens")]:
    print(repr(text), stop, score_completion(text, stop, "02"))

数据划分: {'train': 160, 'dev': 48, 'test': 48}
词表: ['<pad>', '<bos>', '<eos>', 'c', 'o', 'p', 'y', ':', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ',', '=']
回答动作: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '<eos>']
'02' eos {'reward': 1.0, 'reason': 'correct'}
'2' eos {'reward': 0.0, 'reason': 'invalid_format'}
'03' eos {'reward': 0.0, 'reason': 'wrong'}
'02' max_tokens {'reward': 0.0, 'reason': 'missing_eos'}


## 2. 概率从哪里来？先准备真实的因果模型

使用两层、隐藏维度 32、4 个头的字符 Transformer。所有 Dropout 为零。先做 40 次监督预热，再冻结基座，只训练每层 Q/V 的 rank=4 LoRA。

参考模型是 SFT 基座的独立、冻结副本。当前策略与它从相同的分布起步，但后续不能让参考跟着更新。

`eval()` 控制模块行为，不等于 `no_grad()`。采样时可以不保留计算图；**之后必须在仍然相同的参数上，对所采样的序列重新前向并保留 log 概率梯度**。

下面是本课独立使用的全部模型代码。没有外部模型文件，也没有导入上一课代码。

In [5]:
# %% 03 自回归Transformer与Q/V LoRA：不依赖上一课文件
class CausalAttention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.heads, self.head_dim = cfg.heads, cfg.dim // cfg.heads
        self.q_proj = nn.Linear(cfg.dim, cfg.dim, bias=False)
        self.k_proj = nn.Linear(cfg.dim, cfg.dim, bias=False)
        self.v_proj = nn.Linear(cfg.dim, cfg.dim, bias=False)
        self.o_proj = nn.Linear(cfg.dim, cfg.dim, bias=False)

    def forward(self, x: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
        b, t, d = x.shape
        def heads(z):
            return z.reshape(b, t, self.heads, self.head_dim).transpose(1, 2)
        q, k, v = (heads(layer(x)) for layer in [self.q_proj, self.k_proj, self.v_proj])
        causal = torch.ones(t, t, dtype=torch.bool, device=x.device).tril()
        allowed = causal[None, None, :, :] & valid[:, None, None, :]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        weights = F.softmax(scores.masked_fill(~allowed, -torch.inf), dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().reshape(b, t, d)
        return self.o_proj(out)


class TransformerBlock(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.dim)
        self.attn = CausalAttention(cfg)
        self.norm2 = nn.LayerNorm(cfg.dim)
        self.ffn = nn.Sequential(nn.Linear(cfg.dim, cfg.ff_dim), nn.GELU(),
                                 nn.Linear(cfg.ff_dim, cfg.dim))

    def forward(self, x: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x), valid)
        return x + self.ffn(self.norm2(x))


class TinyLM(nn.Module):
    """小型因果Transformer，绝对位置、Pre-LN、GELU；所有Dropout均为零。

    为聚焦策略梯度故意不添加RoPE、GQA、KV cache或量化；不是生产模型缩小复制品。
    forward只返回logits，由外部损失移位一次。输入必须从BOS开始、右侧补齐。
    """
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        cfg.validate()
        self.cfg = cfg
        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.dim, padding_idx=PAD)
        self.position_embedding = nn.Embedding(cfg.max_len, cfg.dim)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.norm = nn.LayerNorm(cfg.dim)
        self.head = nn.Linear(cfg.dim, cfg.vocab_size, bias=False)
        self.apply(self._init_weights)
        with torch.no_grad():
            self.token_embedding.weight[PAD].zero_()

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        require(ids.ndim == 2 and ids.dtype == torch.long, "input_ids需要[B,T] long")
        require(ids.shape[0] > 0 and 0 < ids.shape[1] <= self.cfg.max_len, "输入长度非法")
        require(bool(((ids >= 0) & (ids < self.cfg.vocab_size)).all()), "token越界")
        require(bool((ids[:, 0] == BOS).all()), "本课每条输入从BOS开始，拒绝全PAD")
        valid = ids.ne(PAD)
        require(not bool((valid[:, 1:] & ~valid[:, :-1]).any()), "本课只支持右补齐")
        positions = torch.arange(ids.shape[1], device=ids.device)
        x = self.token_embedding(ids) + self.position_embedding(positions)[None, :, :]
        for block in self.blocks:
            x = block(x, valid)
        return self.head(self.norm(x))


class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int, alpha: float):
        super().__init__()
        require(type(base) is nn.Linear, "只接受未注入的Linear")
        require(type(rank) is int and 0 < rank <= min(base.in_features, base.out_features),
                "rank超出本课允许范围")
        require(math.isfinite(alpha) and alpha > 0, "alpha必须有限为正")
        self.base = base
        self.base.requires_grad_(False)
        self.rank, self.alpha, self.scaling = rank, float(alpha), float(alpha) / rank
        self.enabled = True
        self.lora_A = nn.Parameter(base.weight.new_empty(rank, base.in_features))
        self.lora_B = nn.Parameter(base.weight.new_zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.base(x)
        if self.enabled:
            out = out + self.scaling * F.linear(F.linear(x, self.lora_A), self.lora_B)
        return out


def is_adapter(name: str) -> bool:
    return name.endswith(".lora_A") or name.endswith(".lora_B")


def inject_lora(model: TinyLM, rank: int = 4, alpha: float = 8.0) -> TinyLM:
    require(not any(isinstance(m, LoRALinear) for m in model.modules()), "拒绝重复注入")
    model.requires_grad_(False)
    # SFT刚结束可能留有.grad；清掉它们，才能审计RL阶段是否产生新梯度。
    for p in model.parameters():
        p.grad = None
    for block in model.blocks:
        block.attn.q_proj = LoRALinear(block.attn.q_proj, rank, alpha)
        block.attn.v_proj = LoRALinear(block.attn.v_proj, rank, alpha)
    require(all(p.requires_grad == is_adapter(n) for n, p in model.named_parameters()),
            "可训练集合必须恰好等于Q/V的A/B")
    return model


def freeze_reference(base: TinyLM) -> TinyLM:
    ref = copy.deepcopy(base).eval().requires_grad_(False)
    for p in ref.parameters():
        p.grad = None
    return ref


def base_state(model: TinyLM) -> dict[str, torch.Tensor]:
    return {name.replace(".base.", "."): value
            for name, value in model.state_dict().items() if not is_adapter(name)}


def adapter_snapshot(model: TinyLM) -> dict[str, torch.Tensor]:
    return {name: p.detach().cpu().clone() for name, p in model.named_parameters()
            if is_adapter(name)}


def load_adapter(model: TinyLM, state: dict) -> None:
    expected = {n for n, _ in model.named_parameters() if is_adapter(n)}
    require(set(state) == expected, "adapter键集合不匹配")
    # 先检查全部张量，再修改，避免部分加载。
    params = dict(model.named_parameters())
    for name, value in state.items():
        require(isinstance(value, torch.Tensor) and value.shape == params[name].shape,
                "adapter形状不匹配")
        require(value.is_floating_point() and bool(torch.isfinite(value).all()), "adapter非法数值")
    with torch.no_grad():
        for name, value in state.items():
            params[name].copy_(value.to(params[name]))


@contextmanager
def adapter_disabled(model: TinyLM):
    layers = [m for m in model.modules() if isinstance(m, LoRALinear)]
    previous = [m.enabled for m in layers]
    try:
        for layer in layers:
            layer.enabled = False
        yield
    finally:
        for layer, enabled in zip(layers, previous):
            layer.enabled = enabled

In [6]:
seed_all(59)
demo_base = TinyLM(ModelConfig(vocab_size=len(codec.tokens)))
demo_policy = inject_lora(copy.deepcopy(demo_base))
demo_reference = freeze_reference(demo_base)
print("基座参数:", sum(p.numel() for p in demo_base.parameters()))
print("可训练LoRA:", sum(p.numel() for p in demo_policy.parameters() if p.requires_grad))
print("参考参数可训练:", any(p.requires_grad for p in demo_reference.parameters()))

基座参数: 18944
可训练LoRA: 1024
参考参数可训练: False


## 3. REINFORCE：不对离散评分器求导

对问题 x、回答 y 和可验证奖励 R，目标是：

\[
J(\theta)=\mathbb E_{y\sim\pi_\theta(\cdot|x)}[R(x,y)]
\]

评分器可以是返回 0 或 1 的普通 Python 函数，不必可导。通过对数导数恒等式：

\[
\nabla_\theta J=\mathbb E[R(x,y)\nabla_\theta\log\pi_\theta(y|x)]
\]

因此，样本损失可以写成 `-(reward.detach() * sequence_logp).mean()`。负号是因为优化器做梯度下降，而我们希望最大化奖励。

**离散抽到哪个 token，不需要反向；生成这个 token 的 log 概率需要反向。** PyTorch 的分布文档也用 `sample()` 与 `log_prob()` 展示 REINFORCE 的这种连接。[策略梯度推导](https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html) · [PyTorch distributions](https://docs.pytorch.org/docs/2.10/distributions.html)

下面一起定义：受限回答的 log 概率、留一法和序列级正则的损失。KL 部分稍后展开。

In [7]:
# %% 04 答案策略的概率与奖励梯度

def answer_log_probs(logits: torch.Tensor, codec: CharCodec) -> torch.Tensor:
    """固定合法动作集合的归一化log概率；采样与重评分必须调用同一函数。"""
    require(logits.shape[-1] == len(codec.tokens), "logits词表宽度错误")
    require(bool(torch.isfinite(logits).all()), "模型产生非有限logits")
    return F.log_softmax(logits[..., codec.answer_ids], dim=-1)


def completion_logps(logits: torch.Tensor, labels: torch.Tensor,
                     codec: CharCodec) -> tuple[torch.Tensor, torch.Tensor]:
    require(logits.ndim==3 and labels.shape==logits.shape[:2], "logits/labels形状错误")
    require(labels.dtype==torch.long, "labels需要long")
    require(bool((labels[:,0]==IGNORE).all()), "首位置无前缀预测，必须忽略")
    target = labels[:,1:]
    mask = target.ne(IGNORE)
    require(bool((mask.sum(-1)>0).all()), "每条回答必须有监督/采样位置")
    mapping = labels.new_full((len(codec.tokens),), -1)
    mapping[codec.answer_ids] = torch.arange(len(codec.answer_ids),device=labels.device)
    safe = target.masked_fill(~mask, EOS)
    require(bool(((safe>=0)&(safe<len(codec.tokens))).all()), "标签越界")
    action_ids = mapping[safe]
    require(bool((action_ids>=0).all()), "目标不在固定回答动作集合")
    lp = answer_log_probs(logits[:,:-1,:], codec).gather(-1,action_ids.unsqueeze(-1)).squeeze(-1)
    sequence_lp = lp.masked_fill(~mask,0).sum(-1)
    return sequence_lp, mask.sum(-1)


def leave_one_out(values: torch.Tensor) -> torch.Tensor:
    require(values.ndim==2 and values.shape[1]>=2, "LOO输入需要[B,G]且G>=2")
    require(values.is_floating_point() and bool(torch.isfinite(values).all()), "奖励必须有限浮点")
    v = values.detach()
    baseline = (v.sum(dim=1,keepdim=True)-v)/(v.shape[1]-1)
    return v-baseline


def reinforce_loss(logps: torch.Tensor, ref_logps: torch.Tensor, rewards: torch.Tensor,
                   beta: float, baseline: str = "loo") -> tuple[torch.Tensor, dict]:
    """每组同一prompt的iid采样；每批只在生成参数处更新一次。
    c=R-beta*(logp-logq)，c与baseline均停止梯度。
    这是sequence-level KL正则目标的score-function梯度估计，不是PPO/GRPO。
    """
    require(logps.ndim==2 and logps.shape==ref_logps.shape==rewards.shape, "需要匹配[B,G]")
    require(all(x.is_floating_point() and bool(torch.isfinite(x).all())
                for x in (logps,ref_logps,rewards)), "概率/奖励必须有限")
    require(type(beta) in (int,float) and math.isfinite(beta) and beta>=0, "beta非法")
    require(baseline in {"none","loo"}, "baseline只支持none/loo")
    sampled_log_ratio = logps.detach()-ref_logps.detach()
    shaped = (rewards.detach()-beta*sampled_log_ratio).detach()
    advantage = shaped if baseline=="none" else leave_one_out(shaped)
    loss = -(advantage.detach()*logps).mean()
    info = {"sampled_log_ratio":sampled_log_ratio, "shaped":shaped,
            "advantage":advantage.detach()}
    return loss, info

In [8]:
theta = torch.zeros(3, dtype=torch.float64, requires_grad=True)
logp = F.log_softmax(theta, dim=-1)
# 固定一次已经抽到动作0、且得到奖励1的示例。不是完整训练。
loss = -1.0 * logp[0]
grad = torch.autograd.grad(loss, theta)[0]
print("下降用的梯度:", grad.tolist())
print("更新前概率:", logp.exp().detach().tolist())
print("一次小更新后概率:", F.softmax(theta.detach()-0.1*grad, dim=-1).tolist())

下降用的梯度: [-0.6666666666666667, 0.3333333333333333, 0.3333333333333333]
更新前概率: [0.3333333333333333, 0.3333333333333333, 0.3333333333333333]
一次小更新后概率: [0.3559130712072203, 0.32204346439638987, 0.32204346439638987]


## 4. 回答 log 概率：移位一次，并且沿序列求和

\[
\log\pi(y|x)=\sum_t \log\pi(y_t\mid x,y_{<t})
\]

本模型只输出 logits，不内部移位。因此外部用 `logits[:, :-1]` 预测 `labels[:, 1:]`。

提示不直接累计到回答概率；EOS 是一次真实动作，必须累计；EOS 后的 padding 不参与。目标是完整序列的概率，不要把和换成 token 平均。

本课的固定合法动作集合也应用在这里。**采样时做过过滤或温度调整，却用未经相同处理的概率训练，会改变采样分布与梯度之间的关系。**

In [9]:
rows = splits["train"][:2]
answers = [codec.encode(rows[0]["answer"])+[EOS], [EOS]]
batch = collate_completions(rows, answers, codec, 24)
logits = torch.zeros(*batch["input_ids"].shape, len(codec.tokens), dtype=torch.float64, requires_grad=True)
seq_logp, counts = completion_logps(logits, batch["labels"], codec)
print("回答长度:", counts.tolist())
print("均匀11动作下的序列log概率:", seq_logp.tolist())
print("手算:", [-3*math.log(11), -math.log(11)])
print("这里的EOS参与损失，提示和PAD不参与。")

回答长度: [3, 1]
均匀11动作下的序列log概率: [-7.193685818395112, -2.3978952727983707]
手算: [-7.193685818395112, -2.3978952727983707]
这里的EOS参与损失，提示和PAD不参与。


## 5. Baseline：比较“比预期好多少”

常见估计器用优势 `A=R-b` 替代原奖励。若 baseline 在给定问题后不依赖当前抽样动作，并且不通过它反向，那么：

\[
\mathbb E[b(x)\nabla\log\pi(y|x)] = b(x)\nabla \sum_y \pi(y|x)=0
\]

所以梯度**期望**不变，但方差可能改变。好的 baseline 可以降低方差，差的 baseline 也可能使方差更大；不是加一个减法就必然更稳定。[Baseline 推导](https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html#baselines-in-policy-gradients)

### 留一法：同题独立采样 G 个回答

\[
b_i=\frac{1}{G-1}\sum_{j\ne i}R_j,\qquad A_i=R_i-b_i
\]

每个样本的 baseline 来自**同一问题的其他样本**。保留重复回答，不去重、不只取最佳候选；这才能保持这里使用的独立有放回采样解释。[RLOO 论文](https://arxiv.org/html/2402.14740v1)

组内奖励 `[1,0,0,0]` 时，优势应为 `[1,-1/3,-1/3,-1/3]`。本课没有再除以标准差，不要把它混同于 GRPO 的完整目标。

In [10]:
rewards = torch.tensor([[1.,0.,0.,0.], [0.,0.,0.,0.]], dtype=torch.float64)
print("LOO优势:\n", leave_one_out(rewards))
print("含自身的组均值优势:\n", rewards - rewards.mean(1,keepdim=True))
print("G=4时两者相差缩放4/3；不是相同估计器。")

LOO优势:
 tensor([[ 1.0000, -0.3333, -0.3333, -0.3333],
        [ 0.0000,  0.0000,  0.0000,  0.0000]], dtype=torch.float64)
含自身的组均值优势:
 tensor([[ 0.7500, -0.2500, -0.2500, -0.2500],
        [ 0.0000,  0.0000,  0.0000,  0.0000]], dtype=torch.float64)
G=4时两者相差缩放4/3；不是相同估计器。


## 6. 不用随机抽样碰运气：枚举所有小分布结果

固定三个动作概率 `[0.2,0.5,0.3]`，奖励 `[1,0,0.2]`。每组抽3次，一共只有27种有序组合，可以全部列出，按真实概率加权。

我们比较无 baseline、常数 baseline、包含自身的组均值、留一法，以及一个故意不合适的常数10。

包含自身的均值依赖当前动作。在这个 IID、未标准化的例子中，它让梯度期望变成真实值的 `(G-1)/G`。这里是**确定的缩放偏差**，不是说方向必然完全反转；更不能说任何组内中心化算法都因此不可用，它们的目标与修正需要另行分析。

以下测试针对原始估计器；梯度裁剪和 Adam 是后面的优化操作，不继承“更新轨迹无偏”的保证。

In [11]:
# %% 05 有限分布的精确枚举：无采样噪声核对目标与梯度

def enumerated_baselines(group_size: int = 3) -> dict:
    require(type(group_size) is int and 2<=group_size<=6, "枚举group_size仅限2..6")
    p = torch.tensor([0.2,0.5,0.3],dtype=torch.float64)
    rewards = torch.tensor([1.0,0.0,0.2],dtype=torch.float64)
    truth = p*(rewards-(p*rewards).sum())
    results = {}
    for method in ("none","constant","including_self","loo","bad_constant"):
        expected = torch.zeros_like(p)
        second = 0.0
        for actions in itertools.product(range(3),repeat=group_size):
            ids = torch.tensor(actions)
            prob = float(p[ids].prod())
            r = rewards[ids]
            if method=="none":
                b = torch.zeros_like(r)
            elif method=="constant":
                b = torch.full_like(r,float((p*rewards).sum()))
            elif method=="bad_constant":
                b = torch.full_like(r,10.0)
            elif method=="including_self":
                b = r.mean().expand_as(r)
            else:
                b = (r.sum()-r)/(group_size-1)
            scores = F.one_hot(ids,num_classes=3).to(torch.float64)-p
            estimate = ((r-b).unsqueeze(1)*scores).mean(0)
            expected += prob*estimate
            second += prob*float(estimate.square().sum())
        results[method] = {"expected_gradient":expected.tolist(),
                           "squared_bias":float((expected-truth).square().sum()),
                           "variance_trace":max(0.0,second-float(expected.square().sum()))}
    return {"group_size":group_size,"probabilities":p.tolist(),"rewards":rewards.tolist(),
            "exact_gradient":truth.tolist(),"methods":results}


def exact_kl_gradient_demo(beta: float = 0.3) -> dict:
    theta = torch.tensor([0.2,-0.4,0.7],dtype=torch.float64,requires_grad=True)
    logp = F.log_softmax(theta,dim=0); p=logp.exp()
    logq = torch.tensor([0.4,0.4,0.2],dtype=torch.float64).log()
    reward = torch.tensor([1.0,0.0,0.2],dtype=torch.float64)
    exact_loss = -(p*(reward-beta*(logp-logq))).sum()
    exact = torch.autograd.grad(exact_loss,theta,retain_graph=True)[0]
    shaped = (reward-beta*(logp.detach()-logq)).detach()
    surrogate = -(p.detach()*shaped*logp).sum()
    score_gradient = torch.autograd.grad(surrogate,theta,retain_graph=True)[0]
    bad_surrogate = -(p.detach()*(reward-beta*(logp-logq))*logp).sum()
    bad = torch.autograd.grad(bad_surrogate,theta)[0]
    return {"beta":beta,"exact_gradient":exact.tolist(),
            "stopped_reward_gradient":score_gradient.tolist(),
            "wrong_differentiable_reward_gradient":bad.tolist(),
            "correct_max_error":float((exact-score_gradient).abs().max()),
            "wrong_max_error":float((exact-bad).abs().max())}


def stale_sampling_demo() -> dict:
    old = torch.tensor([0.8,0.2],dtype=torch.float64)
    current = torch.tensor([0.3,0.7],dtype=torch.float64)
    reward = torch.tensor([1.,0.],dtype=torch.float64)
    scores = torch.eye(2,dtype=torch.float64)-current
    truth=(current.unsqueeze(1)*reward.unsqueeze(1)*scores).sum(0)
    stale=(old.unsqueeze(1)*reward.unsqueeze(1)*scores).sum(0)
    weighted=(old.unsqueeze(1)*(current/old).unsqueeze(1)*reward.unsqueeze(1)*scores).sum(0)
    return {"correct_on_policy":truth.tolist(),"stale_unweighted":stale.tolist(),
            "exact_importance_weighted":weighted.tolist(),
            "warning":"本课不实现旧样本复用；每批采样后只更新一次。"}


def mechanisms() -> dict:
    p=torch.tensor([0.8,0.2],dtype=torch.float64)
    q=torch.tensor([0.5,0.5],dtype=torch.float64)
    ratios=p.log()-q.log()
    logits=torch.zeros(3,dtype=torch.float64,requires_grad=True)
    # 固定一个得到奖励1的动作，演示log_prob才是梯度通路。
    sample_loss=-F.log_softmax(logits,dim=-1)[0]
    grad=torch.autograd.grad(sample_loss,logits)[0]
    all_zero=leave_one_out(torch.zeros(2,4,dtype=torch.float64))
    return {
        "one_sample_gradient":grad.tolist(),
        "baselines":enumerated_baselines(3),
        "kl_gradient":exact_kl_gradient_demo(),
        "negative_sample_log_ratio":{"per_action":ratios.tolist(),
            "exact_kl":float((p*ratios).sum()),"note":"单样本和有限批次的log比值可以为负"},
        "zero_reward_group_advantage":all_zero.tolist(),
        "stale_sampling":stale_sampling_demo(),
        "reward_scale_note":"R乘常数而beta不变，会改变相对正则强度。",
    }

In [12]:
mechanism_report = mechanisms()
enum = mechanism_report["baselines"]
print("真实梯度:", enum["exact_gradient"])
for name, data in enum["methods"].items():
    print(name, "平均梯度=", [round(x,6) for x in data["expected_gradient"]],
          "方差迹=", round(data["variance_trace"],8))

真实梯度: [0.148, -0.13, -0.018]
none 平均梯度= [0.148, -0.13, -0.018] 方差迹= 0.05541067
constant 平均梯度= [0.148, -0.13, -0.018] 方差迹= 0.027296
including_self 平均梯度= [0.098667, -0.086667, -0.012] 方差迹= 0.02166163
loo 平均梯度= [0.148, -0.13, -0.018] 方差迹= 0.04873867
bad_constant 平均梯度= [0.148, -0.13, -0.018] 方差迹= 19.10341067


## 7. KL 约束：不要把单样本 log 比值当成确定的 KL

固定参考策略 \(\pi_{\rm ref}\)，加入正则：

\[
J_\beta=\mathbb E_{y\sim\pi_\theta}[R(x,y)]
-\beta D_{\rm KL}(\pi_\theta(\cdot|x)\|\pi_{\rm ref}(\cdot|x))
\]

KL 的作用是惩罚相对于参考的分布偏离。它是**软惩罚**，不是“绝不偏离超过某个阈值”的硬约束，也不自动等价于事实正确或安全。

对当前策略采样的某条回答：

\[
k(y)=\log\pi_\theta(y|x)-\log\pi_{\rm ref}(y|x)
\]

只有在相应分布下取期望，才得到这个方向的 KL。单条或有限批平均可以为负。因此日志字段叫 `mean_sampled_log_ratio`，不冒充精确非负 KL。

定义塑形回报 \(C=R-\beta k\)，然后在组内对 C 做留一法。[序列级正则与 RLOO](https://arxiv.org/html/2402.14740v1) · [官方训练文档](https://huggingface.co/docs/trl/rloo_trainer)

\(\beta\) 与奖励单位有关系：把 R 乘10而不改 \(\beta\)，相对约束就变了。

In [13]:
print(json.dumps(mechanism_report["negative_sample_log_ratio"], ensure_ascii=False, indent=2))
print("例子：抽到第二个动作，log比值约为-0.916；全分布KL仍约为0.193。")

{
  "per_action": [
    0.4700036292457356,
    -0.916290731874155
  ],
  "exact_kl": 0.19274475702175747,
  "note": "单样本和有限批次的log比值可以为负"
}
例子：抽到第二个动作，log比值约为-0.916；全分布KL仍约为0.193。


## 8. 为什么塑形回报要 `detach()`？

本课使用：

```python
shaped = (reward - beta * (logp.detach() - ref_logp.detach())).detach()
advantage = leave_one_out(shaped)
loss = -(advantage * logp).mean()
```

注意：**保留当前 `logp` 的梯度，切断奖励权重与参考路径的梯度。** 不是把整个损失 detach。

因为 R 不显式依赖 θ，且 \(\mathbb E[\nabla\log\pi]=0\)：

\[
\nabla J_\beta
=\mathbb E[(R-\beta\log(\pi/\pi_{ref}))\nabla\log\pi]
\]

如果直接在这个 `-C*logp` 代理里继续对 C 求导，会额外产生不属于上述估计器的项。这里没有再加第二份 KL loss，以免重复计算。

这不表示“所有 KL 正则都要 detach”。如果能对**全分布精确期望**直接求导，那是另一种合法写法。下面将两种正确写法的梯度与错误代理作对照。

In [14]:
kl_check = mechanism_report["kl_gradient"]
print("精确目标梯度:", kl_check["exact_gradient"])
print("停止奖励梯度的枚举估计:", kl_check["stopped_reward_gradient"])
print("两者最大差异:", kl_check["correct_max_error"])
print("错误地对塑形权重也反向，最大差异:", kl_check["wrong_max_error"])

精确目标梯度: [-0.23074165683097062, 0.014107782539471312, 0.2166338742914993]
停止奖励梯度的枚举估计: [-0.2307416568309706, 0.014107782539471307, 0.21663387429149933]
两者最大差异: 2.7755575615628914e-17
错误地对塑形权重也反向，最大差异: 0.05339328724215506


## 9. On-policy：每批只更新一次，然后重新采样

本课固定流程：

```text
同一份当前参数采样 → 不改变参数 → 重算概率 → 一次更新 → 丢弃这批训练样本
```

不把旧样本反复训练多轮，不做 PPO 比率裁剪，不做重要性采样修正。普通策略梯度使用当前策略产生的数据，旧分布采样不能无条件代入。[VPG / on-policy](https://spinningup.openai.com/en/latest/algorithms/vpg.html)

我们保存每条回答采样时的 log 概率，在更新前核对：逐步采样记录与一次性重算结果一致。还记录参数指纹与 `used` 标记，拒绝陈旧样本和第二次更新。

这里的 `policy_before` 是本批行为策略；固定参考模型是 KL 的锚点。**“旧策略”与“参考策略”不是一个角色。**

In [15]:
print(json.dumps(mechanism_report["stale_sampling"], ensure_ascii=False, indent=2))
print("重要性加权在小分布上能校正；本课训练器选择更简单的重新采样。")

{
  "correct_on_policy": [
    0.21,
    -0.21
  ],
  "stale_unweighted": [
    0.5599999999999999,
    -0.5599999999999999
  ],
  "exact_importance_weighted": [
    0.21,
    -0.21
  ],
  "warning": "本课不实现旧样本复用；每批采样后只更新一次。"
}
重要性加权在小分布上能校正；本课训练器选择更简单的重新采样。


In [16]:
# %% 06 按当前策略采样；保存行为概率但不保留采样计算图

@contextmanager
def eval_mode(model: nn.Module):
    old=model.training
    try:
        model.eval()
        yield
    finally:
        model.train(old)


@torch.no_grad()
def sample_rollouts(policy: TinyLM, rows: list[dict], codec: CharCodec,
                    group_size: int, generator: torch.Generator,
                    horizon: int = 3, greedy: bool = False) -> dict:
    require(bool(rows) and type(group_size) is int and group_size>0, "空输入/非法组大小")
    require(type(horizon) is int and 1<=horizon<=3, "horizon必须1..3")
    expanded=[copy.deepcopy(row) for row in rows for _ in range(group_size)]
    prefixes=[[BOS]+codec.encode(row["prompt"]) for row in expanded]
    require(len({len(x) for x in prefixes})==1, "本课采样提示固定等长，不支持左补齐混用")
    require(len(prefixes[0])+horizon<=policy.cfg.max_len,"生成预算超过上下文")
    device=next(policy.parameters()).device
    ids=torch.tensor(prefixes,dtype=torch.long,device=device)
    active=torch.ones(len(expanded),dtype=torch.bool,device=device)
    answers=[[] for _ in expanded]; behavior=torch.zeros(len(expanded),dtype=torch.float64,device=device)
    with eval_mode(policy):
        for _ in range(horizon):
            lp=answer_log_probs(policy(ids)[:,-1,:],codec)
            action=lp.argmax(-1) if greedy else torch.multinomial(lp.exp(),1,generator=generator).squeeze(1)
            token_table=torch.tensor(codec.answer_ids,device=device)
            next_ids=token_table[action]
            chosen_lp=lp.gather(1,action[:,None]).squeeze(1)
            behavior+=chosen_lp.to(torch.float64)*active
            old_active=active.clone()
            for i in range(len(expanded)):
                if bool(old_active[i]):
                    answers[i].append(int(next_ids[i]))
            active=active & next_ids.ne(EOS)
            # 已结束行追加PAD。仍活跃行的计算不受其他batch行影响。
            append=next_ids.masked_fill(~old_active,PAD)
            ids=torch.cat([ids,append[:,None]],dim=1)
            if not bool(active.any()):
                break
    samples=[]
    for i,(row,ans) in enumerate(zip(expanded,answers)):
        stopped=ans[-1]==EOS
        text=codec.decode_answer(ans[:-1] if stopped else ans)
        scoring=score_completion(text,"eos" if stopped else "max_tokens",row["answer"])
        samples.append({"task_id":row["id"],"group_index":i//group_size,"sample_index":i%group_size,
                        "prompt":row["prompt"],"token_ids":ans,"text":text,
                        "stop_reason":"eos" if stopped else "max_tokens",
                        "behavior_logp":float(behavior[i]),**scoring})
    return {"rows":expanded,"answers":answers,"samples":samples,
            "prompt_count":len(rows),"group_size":group_size,
            "policy_hash":tensor_hash(policy.state_dict()),"codec_hash":digest(codec.contract()),
            "horizon":horizon,"greedy":greedy,"used":False}

In [17]:
small_rows = splits["train"][:2]
rollout = sample_rollouts(demo_policy, small_rows, codec, group_size=4,
                         generator=torch.Generator().manual_seed(59))
for item in rollout["samples"]:
    print(item["task_id"], item["sample_index"], repr(item["text"]),
          item["stop_reason"], "reward=", item["reward"])
print("这些是未预热模型的实际样本，没有替换为参考答案。")

copy_02_06 0 '751' max_tokens reward= 0.0
copy_02_06 1 '39' eos reward= 0.0
copy_02_06 2 '7' eos reward= 0.0
copy_02_06 3 '348' max_tokens reward= 0.0
copy_07_13 0 '303' max_tokens reward= 0.0
copy_07_13 1 '156' max_tokens reward= 0.0
copy_07_13 2 '315' max_tokens reward= 0.0
copy_07_13 3 '' eos reward= 0.0
这些是未预热模型的实际样本，没有替换为参考答案。


## 10. 稀疏奖励与 SFT 起点

如果 G 个回答都错，在 \(\beta=0\) 的留一法里，原奖励与优势都为0，这一组不会提供任务梯度。采样本身不会保证凭空发现正确算法。

本课先做40次SFT，以便模型有一定概率生成合格回答。SFT和RL只接收训练侧；开发集用于观察，不选择本课轮次；测试集在所有分支最后一步保存后才评测。

所有组的 G 都固定为4，每题等权，完成序列等权。对无EOS的 cap 样本仍保留其真实概率与0奖励，不删除失败样本，也不重新抽到成功为止。

In [18]:
def sft_warmup(base: TinyLM, rows: list[dict], codec: CharCodec, cfg: TrainConfig) -> list[dict]:
    require(all(row["split"]=="train" for row in rows),"SFT只接收训练侧")
    optimizer=torch.optim.Adam(base.parameters(),lr=cfg.sft_lr)
    rng=random.Random(cfg.seed+1); history=[]
    device=next(base.parameters()).device
    base.train()
    for step in range(cfg.sft_steps):
        batch_rows=rng.sample(rows, min(32,len(rows)))
        answers=[codec.encode(r["answer"])+[EOS] for r in batch_rows]
        batch=collate_completions(batch_rows,answers,codec,base.cfg.max_len,device)
        optimizer.zero_grad(set_to_none=True)
        lp,counts=completion_logps(base(batch["input_ids"]),batch["labels"],codec)
        loss=-lp.sum()/counts.sum()
        loss.backward()
        nn.utils.clip_grad_norm_(base.parameters(),cfg.max_grad_norm,error_if_nonfinite=True)
        optimizer.step()
        if step==0 or (step+1)%10==0 or step+1==cfg.sft_steps:
            history.append({"step":step+1,"token_nll":float(loss.detach())})
    base.eval()
    for p in base.parameters(): p.grad=None
    return history

## 11. 一次真实的 LoRA 策略更新

更新前先检查样本身份、动作集合、行为概率与训练划分；然后清零梯度、重算 log 概率、冻结参考、塑形与LOO、反向、裁剪、更新。

梯度裁剪阈值为1，目的是限制实际更新的数值尺度，不是 PPO 的概率比裁剪。`surrogate_loss` 只是当前样本上用于反向传播的代理，**不同批次的值不能单独作为能力成绩**。

保存所有训练采样文本、奖励原因和优势，便于检查“训练信号是否来自真实回答”。这份文件包含训练标签，不要把它作为未知测试数据的来源。

In [19]:
def one_on_policy_step(policy: TinyLM, reference: TinyLM, rollout: dict,
                       codec: CharCodec, optimizer: torch.optim.Optimizer,
                       beta: float, baseline: str, clip: float=1.0) -> dict:
    require(not rollout["used"],"同一批样本已用于更新，不能直接再次当on-policy数据")
    require(not rollout["greedy"],"贪心样本不是本课categorical on-policy数据")
    require(rollout["policy_hash"]==tensor_hash(policy.state_dict()),"参数已改变，拒绝陈旧样本")
    require(rollout["codec_hash"]==digest(codec.contract()),"采样动作空间/分词契约变了")
    require(all(row["split"]=="train" for row in rollout["rows"]),"训练入口拒绝dev/test")
    require(all(not p.requires_grad for p in reference.parameters()),"参考必须冻结")
    require({id(p) for g in optimizer.param_groups for p in g["params"]}==
            {id(p) for p in policy.parameters() if p.requires_grad},"优化器参数对象不匹配")
    require(all(p.requires_grad==is_adapter(n) for n,p in policy.named_parameters()),"只允许更新Q/V LoRA")
    require(type(clip) in (int,float) and math.isfinite(clip) and clip>0,"clip非法")
    b,g=rollout["prompt_count"],rollout["group_size"]
    require(len(rollout["samples"])==b*g,"样本分组数量错误")
    for i, sample in enumerate(rollout["samples"]):
        row=rollout["rows"][i]; ans=rollout["answers"][i]
        require(sample["task_id"]==row["id"] and sample["token_ids"]==ans,"样本错配")
        require(sample["group_index"]==i//g and sample["sample_index"]==i%g,"组顺序改变")
        require(row["id"]==rollout["rows"][(i//g)*g]["id"],"同组必须是同一个prompt")
        text=codec.decode_answer(ans[:-1] if ans[-1]==EOS else ans)
        stop="eos" if ans[-1]==EOS else "max_tokens"
        require(sample["text"]==text and sample["stop_reason"]==stop,"文本/停止状态不符")
        require(sample["reward"]==score_completion(text,stop,row["answer"])["reward"],"奖励与当前响应不匹配")
    device=next(policy.parameters()).device
    batch=collate_completions(rollout["rows"],rollout["answers"],codec,policy.cfg.max_len,device)
    optimizer.zero_grad(set_to_none=True)
    # eval只关闭随机行为，不关闭autograd。
    with eval_mode(policy), eval_mode(reference):
        lp,_=completion_logps(policy(batch["input_ids"]),batch["labels"],codec)
        with torch.no_grad():
            ref_lp,_=completion_logps(reference(batch["input_ids"]),batch["labels"],codec)
        old=torch.tensor([s["behavior_logp"] for s in rollout["samples"]],device=device,dtype=lp.dtype)
        error=float((lp.detach()-old).abs().max())
        require(torch.allclose(lp.detach(),old,rtol=2e-5,atol=2e-5),"采样和重算概率不同：检查温度/过滤/移位/Dropout")
        rewards=torch.tensor([s["reward"] for s in rollout["samples"]],device=device,dtype=lp.dtype).reshape(b,g)
        loss,info=reinforce_loss(lp.reshape(b,g),ref_lp.reshape(b,g),rewards,beta,baseline)
        loss.backward()
        params=[p for p in policy.parameters() if p.requires_grad]
        require(all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in params),"adapter梯度缺失或非有限")
        norm=nn.utils.clip_grad_norm_(params,clip,error_if_nonfinite=True)
        optimizer.step()
    rollout["used"]=True
    for i,sample in enumerate(rollout["samples"]):
        sample.update({"reference_logp":float(ref_lp[i]),
                       "sampled_log_ratio":float(info["sampled_log_ratio"].flatten()[i]),
                       "shaped_reward":float(info["shaped"].flatten()[i]),
                       "advantage":float(info["advantage"].flatten()[i])})
    return {"surrogate_loss":float(loss.detach()), "mean_reward":float(rewards.mean()),
            "mean_sampled_log_ratio":float(info["sampled_log_ratio"].mean()),
            "mean_shaped_reward":float(info["shaped"].mean()),
            "mean_abs_advantage":float(info["advantage"].abs().mean()),
            "zero_raw_reward_groups":int((rewards.sum(1)==0).sum()),
            "gradient_norm_before_clip":float(norm),"behavior_recompute_max_error":error,
            "completions":b*g,"generated_tokens":sum(len(a) for a in rollout["answers"])}

## 12. 为什么这里可以计算“精确采样成功概率”？

对本任务，每题只有一种被接受的完整轨迹：**目标两位字符串＋EOS**。三步视野刚好容纳它，所以：

\[
P(\text{成功}|x)=\pi_\theta(\text{两位答案},EOS|x)
\]

通过目标序列的概率乘积可以直接算出，不依赖有限采样是否刚好抽到它。它仍基于本课的固定回答动作集合，不是通用问答正确率。

同时报告每题8次实际采样的成功数，以及贪心生成的正确题数。三者不能混用：贪心选择改变了生成规则，而有限采样有波动。

多解数学任务通常不能这样用单条参考答案概率表示全部成功概率。

In [20]:
# %% 07 固定评测：精确成功概率、实际采样成功、贪心成功分别报告

@torch.no_grad()
def evaluate_policy(model: TinyLM, rows: list[dict], codec: CharCodec,
                    sample_count: int=8, seed: int=90059) -> dict:
    require(bool(rows),"评测不能为空")
    device=next(model.parameters()).device
    answers=[codec.encode(r["answer"])+[EOS] for r in rows]
    batch=collate_completions(rows,answers,codec,model.cfg.max_len,device)
    with eval_mode(model):
        target_lp,_=completion_logps(model(batch["input_ids"]),batch["labels"],codec)
    # 本任务只有一种接受的完整轨迹(answer+EOS)，故exp(logp)就是它的精确成功概率。
    probabilities=target_lp.exp().cpu()
    gen=torch.Generator(device=device).manual_seed(seed)
    sampled=sample_rollouts(model,rows,codec,sample_count,gen)
    greedy=sample_rollouts(model,rows,codec,1,gen,greedy=True)
    return {"tasks":len(rows),"exact_success_probability":float(probabilities.mean()),
            "mean_correct_sequence_nll":float(-target_lp.mean()),
            "sampled_successes":sum(s["reward"] for s in sampled["samples"]),
            "sampled_total":len(sampled["samples"]),
            "sampled_success_rate":sum(s["reward"] for s in sampled["samples"])/len(sampled["samples"]),
            "greedy_successes":sum(s["reward"] for s in greedy["samples"]),
            "per_task_probability":[{"id":r["id"],"p":float(p)} for r,p in zip(rows,probabilities)],
            "greedy":greedy["samples"],"samples":sampled["samples"],
            "note":"小型同语法复制任务；概率针对本课固定动作集合和三步视野。"}


def compact_eval(report: dict) -> dict:
    return {k:v for k,v in report.items() if k not in {"per_task_probability","greedy","samples"}}

## 13. 三条训练分支与固定预算

| 分支 | 任务奖励 | Baseline | KL系数 |
|---|---|---|---:|
| `plain` | 精确0/1 | 无 | 0 |
| `loo` | 精确0/1 | 同题留一法 | 0 |
| `loo_kl` | 精确0/1 | 对塑形回报留一 | 0.2 |

三条分支从**同一SFT基座、同一LoRA初始化、同一prompt调度表**开始。每条固定60次更新，每次8道题，每题4个候选，共 **1,920条训练完成序列/分支**。

策略改变后，后续实际回答也会不同。共享随机数种子不意味着各分支永远使用相同文本。生成token数量也会因EOS时机变化，不宣称等时延或等GPU计算量。

RL只更新1,024个LoRA参数。基座18,944个参数，全部冻结；参考是单独冻结副本。训练记录中的生成量仅计RL采样，SFT、开发集诊断与最终评测另有开销。

固定保留最后一步，不根据测试成绩重新选轮。单种子、小任务不能用来给REINFORCE、LOO与KL作普遍排名。

In [21]:
def run_experiment(output_root: Path=Path("lesson_59_outputs"),
                   cfg: TrainConfig|None=None, device_name="cpu", mode="all") -> Path:
    cfg=cfg or TrainConfig(); cfg.validate()
    device=resolve_device(device_name)
    require(mode in {"all","plain","loo","loo_kl"},"模式非法")
    stamp=datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S_%f")
    out=Path(output_root)/stamp; out.mkdir(parents=True,exist_ok=False)
    seed_all(cfg.seed)
    codec=CharCodec(); splits=make_splits(cfg.seed); validate_splits(splits)
    model_cfg=ModelConfig(vocab_size=len(codec.tokens))
    manifest={"version":VERSION,"python":platform.python_version(),"torch":torch.__version__,
              "device":str(device),"cuda_available":torch.cuda.is_available(),
              "status":"running","config":asdict(cfg),"model":asdict(model_cfg),
              "sources":SOURCES,"limitations":["single_seed_synthetic_task","not_pretrained_LLM",
                "no_PPO_GRPO_no_importance_reuse","no_GPU_performance_claim",
                "fixed_answer_alphabet_temperature_1","no_exact_resume_state"]}
    write_json(out/"manifest.json",manifest)
    try:
        write_json(out/"mechanisms.json",mechanisms())
        write_json(out/"split.json",splits)
        write_json(out/"tokenizer.json",codec.contract())
        base=TinyLM(model_cfg).to(device)
        print("SFT warmup...",flush=True)
        warmup=sft_warmup(base,splits["train"],codec,cfg)
        reference=freeze_reference(base)
        reference_hash=tensor_hash(reference.state_dict())
        torch.save({"version":VERSION,"model_config":asdict(model_cfg),"codec":codec.contract(),
                    "state":{n:t.cpu() for n,t in reference.state_dict().items()},
                    "reference_hash":reference_hash},out/"reference.pt")
        template=inject_lora(copy.deepcopy(base),cfg.rank,cfg.alpha).to(device).eval()
        schedule_rng=random.Random(cfg.seed+44)
        schedule=[schedule_rng.sample(splits["train"],cfg.prompts_per_update) for _ in range(cfg.updates)]
        write_json(out/"schedule.json",[[r["id"] for r in rows] for rows in schedule])
        summary={"base_parameters":sum(p.numel() for p in base.parameters()),
                 "trainable_lora_parameters":sum(p.numel() for p in template.parameters() if p.requires_grad),
                 "sft":warmup,"reference_hash":reference_hash,"arms":{}}
        # Dev可以观察训练，不用于本课选轮；所有分支固定最后一步后才评测test。
        summary["reference_dev"]=compact_eval(evaluate_policy(reference,splits["dev"],codec,cfg.eval_samples))
        methods={"plain":(0.0,"none"),"loo":(0.0,"loo"),"loo_kl":(cfg.kl_beta,"loo")}
        if mode!="all": methods={mode:methods[mode]}
        trained={}
        for name,(beta,baseline) in methods.items():
            print("On-policy arm:",name,"beta:",beta,flush=True)
            policy=copy.deepcopy(template)
            optimizer=torch.optim.Adam([p for p in policy.parameters() if p.requires_grad],lr=cfg.rl_lr)
            generator=torch.Generator(device=device).manual_seed(cfg.seed+100)
            history=[]; dev_history=[]
            with (out/(name+"_rollouts.jsonl")).open("w",encoding="utf-8") as log:
                for update,rows in enumerate(schedule,1):
                    rollout=sample_rollouts(policy,rows,codec,cfg.group_size,generator,cfg.max_new_tokens)
                    record=one_on_policy_step(policy,reference,rollout,codec,optimizer,beta,baseline,cfg.max_grad_norm)
                    record["update"]=update; history.append(record)
                    log.write(canonical({"update":update,"policy_before":rollout["policy_hash"],
                                         "samples":rollout["samples"]})+"\n")
                    if update==1 or update%20==0 or update==cfg.updates:
                        dev=evaluate_policy(policy,splits["dev"],codec,4,seed=90159)
                        dev_history.append({"update":update,**compact_eval(dev)})
                        print(f"  step {update}: reward={record['mean_reward']:.3f}, dev P(correct)={dev['exact_success_probability']:.4f}",flush=True)
            require(tensor_hash(base_state(policy))==reference_hash,"基座权重发生改变")
            require(tensor_hash(reference.state_dict())==reference_hash,"参考权重发生改变")
            require(all(p.grad is None for p in reference.parameters()),"参考收到梯度")
            checkpoint={"version":VERSION,"config":asdict(cfg),"arm":name,"beta":beta,
                        "baseline":baseline,"reference_hash":reference_hash,
                        "codec":codec.contract(),"adapter":adapter_snapshot(policy),
                        "selection":"fixed_last_update_no_test_selection"}
            torch.save(checkpoint,out/(name+".pt"))
            summary["arms"][name]={"beta":beta,"baseline":baseline,
                "history":history,"dev_history":dev_history,"base_unchanged":True,
                "completions":sum(r["completions"] for r in history),
                "generated_tokens":sum(r["generated_tokens"] for r in history),
                "final_policy_hash":tensor_hash(policy.state_dict())}
            trained[name]=policy.cpu()
        # 已保存固定最后一步，才首次打开test比较。没有test选超参/选轮。
        reference=reference.to(device)
        evaluation={"reference":evaluate_policy(reference,splits["test"],codec,cfg.eval_samples)}
        for name,policy in trained.items():
            policy=policy.to(device)
            evaluation[name]=evaluate_policy(policy,splits["test"],codec,cfg.eval_samples)
            summary["arms"][name]["test"]=compact_eval(evaluation[name])
        summary["reference_test"]=compact_eval(evaluation["reference"])
        write_json(out/"evaluation.json",evaluation)
        write_json(out/"training.json",summary)
        reload_report=reload_run(out,device_name)
        write_json(out/"reload.json",reload_report)
        manifest["status"]="completed"; write_json(out/"manifest.json",manifest)
        print("Outputs:",out,flush=True)
        return out
    except Exception as exc:
        manifest.update(status="failed",error=type(exc).__name__+": "+str(exc))
        write_json(out/"manifest.json",manifest)
        raise


def reload_run(directory: Path, device_name="cpu") -> dict:
    directory=Path(directory); device=resolve_device(device_name)
    blob=torch.load(directory/"reference.pt",map_location="cpu",weights_only=True)
    require(blob["version"]==VERSION,"检查点版本不匹配")
    codec=CharCodec(); require(blob["codec"]==codec.contract(),"分词/采样契约不匹配")
    base=TinyLM(ModelConfig(**blob["model_config"])).to(device)
    base.load_state_dict(blob["state"],strict=True); base.eval().requires_grad_(False)
    require(tensor_hash(base.state_dict())==blob["reference_hash"],"参考文件内容不匹配")
    saved=json.loads((directory/"evaluation.json").read_text(encoding="utf-8"))
    splits=json.loads((directory/"split.json").read_text(encoding="utf-8")); validate_splits(splits)
    results={}
    for name in sorted(set(saved)-{"reference"}):
        ck=torch.load(directory/(name+".pt"),map_location="cpu",weights_only=True)
        require(ck["version"]==VERSION and ck["reference_hash"]==blob["reference_hash"],"adapter接错参考")
        require(ck["codec"]==codec.contract(),"adapter动作集合不匹配")
        model=inject_lora(copy.deepcopy(base),ck["config"]["rank"],ck["config"]["alpha"])
        load_adapter(model,ck["adapter"]); model.eval().requires_grad_(False)
        report=evaluate_policy(model,splits["test"],codec,ck["config"]["eval_samples"])
        p1=torch.tensor([r["p"] for r in report["per_task_probability"]])
        p0=torch.tensor([r["p"] for r in saved[name]["per_task_probability"]])
        same_ids=[r["token_ids"] for r in report["greedy"]]==[r["token_ids"] for r in saved[name]["greedy"]]
        error=float((p1-p0).abs().max())
        require(error<2e-5 and same_ids,"同设备重载结果不一致；跨后端不承诺逐位相同")
        results[name]={"probability_max_error":error,"greedy_ids_equal":same_ids,
                       "trainable_parameters":sum(p.numel() for p in model.parameters() if p.requires_grad)}
    return {"arms":results,"mode":"inference_only_not_exact_training_resume"}

In [22]:
RUN_TRAINING = True
RUN_DIR = None
if RUN_TRAINING:
    RUN_DIR = run_experiment(OUTPUT_ROOT, TrainConfig(), DEVICE, mode="all")
else:
    print("已跳过完整训练；数学与独立练习仍可继续。")

SFT warmup...


On-policy arm: plain beta: 0.0


  step 1: reward=0.094, dev P(correct)=0.3346


  step 20: reward=0.250, dev P(correct)=0.3423


  step 40: reward=0.375, dev P(correct)=0.3457


  step 60: reward=0.438, dev P(correct)=0.3476


... (7 outputs left out)

  step 20: reward=0.250, dev P(correct)=0.3368


  step 40: reward=0.375, dev P(correct)=0.3378


  step 60: reward=0.438, dev P(correct)=0.3417


Outputs: lesson_59_outputs/run_20261103_025432_284484


## 14. 阅读实验报告，而不是只看最后一个 loss

下面的表由本次实际运行结果打印，没有预填成功分数。主要关注：

- 精确成功概率是否改变？贪心结果是否出现改善或退步？
- 有限采样结果是否与精确概率存在波动？
- 三条分支的训练完成数量相同，生成token是否相同？

同一批次的 log 比值平均不是精确全分布 KL；一次报告负值也不自动表示实现错误。真正数学性质由前面的全分布枚举核对。

本课不是困难数学推理实验：它是已知正确规则的复制任务，用来确认真实 on-policy 更新链路。

In [23]:
if RUN_DIR is not None:
    training_report = json.loads((RUN_DIR/"training.json").read_text(encoding="utf-8"))
    evaluation_report = json.loads((RUN_DIR/"evaluation.json").read_text(encoding="utf-8"))
    print("方法       精确P(正确)   采样成功/总数   贪心正确/题数")
    for name, report in evaluation_report.items():
        print(f"{name:10} {report['exact_success_probability']:.6f}       "
              f"{int(report['sampled_successes'])}/{report['sampled_total']}        "
              f"{int(report['greedy_successes'])}/{report['tasks']}")
    print("训练采样开销：")
    for name, report in training_report["arms"].items():
        print(name, report["completions"], "条回答，", report["generated_tokens"], "个输出token")

方法       精确P(正确)   采样成功/总数   贪心正确/题数
loo        0.256219       99/384        33/48
loo_kl     0.257316       99/384        34/48
plain      0.262651       99/384        32/48
reference  0.250931       97/384        29/48
训练采样开销：
loo 1920 条回答， 5583 个输出token
loo_kl 1920 条回答， 5581 个输出token
plain 1920 条回答， 5563 个输出token


In [24]:
if RUN_DIR is not None:
    print("按测试集原顺序展示前4题，不筛选成功例子：")
    for i in range(4):
        before = evaluation_report["reference"]["greedy"][i]
        after = evaluation_report["loo_kl"]["greedy"][i]
        print(before["prompt"], "基座=", repr(before["text"]), before["reward"],
              "LOO+KL=", repr(after["text"]), after["reward"])

按测试集原顺序展示前4题，不筛选成功例子：
copy:13,07= 基座= '03' 0.0 LOO+KL= '07' 1.0
copy:09,12= 基座= '12' 1.0 LOO+KL= '12' 1.0
copy:09,00= 基座= '00' 1.0 LOO+KL= '00' 1.0
copy:15,14= 基座= '13' 0.0 LOO+KL= '13' 0.0


## 15. 保存、冻结检查与重载

完整运行保存：

```text
reference.pt             SFT参考、词表与模型配置
plain.pt / loo.pt / loo_kl.pt
*_rollouts.jsonl          每一步的实际采样、log概率、奖励和优势
training.json            SFT及三分支更新记录
split.json / schedule.json
mechanisms.json / evaluation.json / reload.json
manifest.json
```

代码逐分支检查冻结基座与参考的内容哈希不变，参考没有梯度；再在干净基座上注入相同结构、加载adapter，核对概率与贪心编号。

新进程命令：

```bash
python lesson_59_reinforce_baselines_and_kl.py --reload-run 你的实际运行目录
```

这是推理恢复，不包含精确续训所需的完整状态。只加载可信本地文件。

In [25]:
if RUN_DIR is not None:
    print(json.dumps(reload_run(RUN_DIR, DEVICE), ensure_ascii=False, indent=2))

{
  "arms": {
    "loo": {
      "probability_max_error": 0.0,
      "greedy_ids_equal": true,
      "trainable_parameters": 0
    },
    "loo_kl": {
      "probability_max_error": 0.0,
      "greedy_ids_equal": true,
      "trainable_parameters": 0
    },
    "plain": {
      "probability_max_error": 0.0,
      "greedy_ids_equal": true,
      "trainable_parameters": 0
    }
  },
  "mode": "inference_only_not_exact_training_resume"
}


## 16. 自动检查

检查包括：奖励与响应一致、EOS与padding、概率归一化、梯度方向、LOO期望、KL梯度、陈旧样本、冻结参数、有限视野概率总和与磁盘重载。

这些检查验证本课具体约定，不是预训练模型能力、安全性或GPU性能认证。数值差异也受版本、设备与算子影响，不保证跨平台逐位复现。[PyTorch reproducibility](https://docs.pytorch.org/docs/2.10/notes/randomness.html)

In [26]:
# %% 08 自动检查：数学、数据、采样、梯度与冻结状态
class Checks:
    def __init__(self):
        self.names=[]

    def ok(self,name: str,condition) -> None:
        if not bool(condition):
            raise AssertionError(name)
        self.names.append(name)

    def close(self,name: str,a,b,atol=1e-10,rtol=1e-10) -> None:
        self.ok(name,torch.allclose(torch.as_tensor(a,dtype=torch.float64),
                                   torch.as_tensor(b,dtype=torch.float64),atol=atol,rtol=rtol))

    def raises(self,name: str,fn: Callable) -> None:
        try:
            fn()
        except (LessonError, ValueError):
            self.names.append(name)
            return
        raise AssertionError(name+": 应明确拒绝")


def run_checks() -> dict:
    seed_all(5900)
    c=Checks(); codec=CharCodec(); splits=make_splits()
    validate_splits(splits)
    c.ok("固定划分160/48/48",[len(splits[k]) for k in ("train","dev","test")]==[160,48,48])
    c.ok("二十项词表与十一种回答动作",len(codec.tokens)==20 and len(codec.answer_ids)==11)
    c.ok("prompt等长",len({len(r['prompt']) for rows in splits.values() for r in rows})==1)
    c.ok("模板不需要chosen/rejected",all('chosen' not in r and 'rejected' not in r for r in splits['train']))
    c.ok("精确答案且EOS奖励1",score_completion("02","eos","02")["reward"]==1)
    c.ok("前导零是任务的一部分",score_completion("2","eos","02")["reward"]==0)
    c.ok("错误数值奖励0",score_completion("03","eos","02")["reason"]=="wrong")
    c.ok("即使字面正确缺EOS也奖0",score_completion("02","max_tokens","02")["reward"]==0)
    c.ok("空回答奖0",score_completion("","eos","02")["reward"]==0)
    c.raises("执行错误不能伪装低奖励",lambda:score_completion("02","runtime_error","02"))
    c.raises("错误参考标签拒绝",lambda:score_completion("2","eos","2"))
    c.raises("未知字符拒绝",lambda:codec.encode("未知"))
    rows=splits['train'][:2]
    answers=[codec.encode(rows[0]['answer'])+[EOS],[EOS]]
    batch=collate_completions(rows,answers,codec,24)
    prefix_len=1+len(rows[0]['prompt'])
    c.ok("提示标签忽略",bool((batch['labels'][:,:prefix_len]==IGNORE).all()))
    c.ok("EOS参与log概率",int((batch['labels']==EOS).sum())==2)
    c.ok("只补右侧PAD",bool((batch['input_ids'][1,-2:]==PAD).all()))
    c.ok("PAD标签忽略",bool((batch['labels'][1,-2:]==IGNORE).all()))
    c.raises("不允许EOS后继续回答",lambda:collate_completions(rows[:1],[[EOS,codec.to_id['1']]],codec,24))
    c.raises("拒绝静默长度截断",lambda:collate_completions(rows,answers,codec,13))
    c.raises("拒绝非回答动作",lambda:collate_completions(rows[:1],[[BOS]],codec,24))
    logits=torch.zeros(2,batch['input_ids'].shape[1],20,dtype=torch.float64,requires_grad=True)
    lp,n=completion_logps(logits,batch['labels'],codec)
    c.close("log概率按有效回答token求和",lp,[-3*math.log(11),-math.log(11)])
    c.close("统计有效回答长度",n,[3,1])
    grad=torch.autograd.grad(lp.sum(),logits)[0]
    c.ok("非回答词的输出logits梯度为零",bool((grad[:,:,codec.to_id['c']]==0).all()))
    c.ok("无关早期prompt logits梯度为零",bool((grad[:,:prefix_len-1]==0).all()))
    c.ok("预测首个回答的位置有梯度",float(grad[:,prefix_len-1].abs().sum())>0)
    c.raises("没有目标的批次拒绝",lambda:completion_logps(logits,batch['labels'].new_full(batch['labels'].shape,IGNORE),codec))
    c.raises("非long标签拒绝",lambda:completion_logps(logits,batch['labels'].float(),codec))
    c.raises("非有限logits拒绝",lambda:answer_log_probs(torch.full((1,20),float('nan')),codec))
    c.close("动作集合概率归一化",answer_log_probs(torch.randn(2,3,20,dtype=torch.float64),codec).exp().sum(-1),torch.ones(2,3))

    values=torch.tensor([[1.,0.,0.,0.],[2.,2.,2.,2.]],dtype=torch.float64,requires_grad=True)
    adv=leave_one_out(values)
    c.close("LOO排除自身",adv[0],[1.,-1/3,-1/3,-1/3])
    c.close("每组相同回报优势为零",adv[1],torch.zeros(4))
    c.close("每组优势求和为零",adv.sum(1),[0.,0.])
    c.ok("优势停止梯度",not adv.requires_grad)
    c.close("常量平移不改LOO",leave_one_out(values+12),adv)
    c.raises("单样本LOO拒绝",lambda:leave_one_out(torch.zeros(1,1)))
    c.raises("奖励NaN拒绝",lambda:leave_one_out(torch.tensor([[1.,float('nan')]])))
    enum=enumerated_baselines(3)
    truth=enum['exact_gradient']
    for key in ('none','constant','loo'):
        c.close(key+"枚举梯度无偏",enum['methods'][key]['expected_gradient'],truth)
    c.close("含自身均值产生2/3缩放",enum['methods']['including_self']['expected_gradient'],torch.tensor(truth,dtype=torch.float64)*2/3)
    c.ok("LOO不保证消除方差",enum['methods']['loo']['variance_trace']>0)
    c.ok("不合适常数可能大幅增加方差",enum['methods']['bad_constant']['variance_trace']>100*enum['methods']['none']['variance_trace'])
    c.close("不合适常数仍保持期望",enum['methods']['bad_constant']['expected_gradient'],truth)
    x=torch.tensor([[-1.,-2.,-3.]],dtype=torch.float64,requires_grad=True)
    ref=torch.tensor([[-2.,-2.,-2.]],dtype=torch.float64,requires_grad=True)
    reward=torch.tensor([[1.,0.,0.]],dtype=torch.float64,requires_grad=True)
    loss,info=reinforce_loss(x,ref,reward,0.2,'loo')
    gx,gr,gw=torch.autograd.grad(loss,(x,ref,reward),allow_unused=True)
    c.close("损失梯度是负优势除样本数",gx,-info['advantage']/3)
    c.ok("参考log概率没有梯度",gr is None)
    c.ok("奖励张量没有梯度",gw is None)
    c.ok("塑形奖励停止梯度",not info['shaped'].requires_grad)
    c.close("KL惩罚方向",info['shaped'],[[0.8,0.,0.2]])
    c.ok("单样本log比值允许负数",float(info['sampled_log_ratio'][0,2])<0)
    none_loss,none_info=reinforce_loss(x,ref,reward,0.,'none')
    c.close("无baseline直接使用原奖励",none_info['advantage'],reward.detach())
    c.raises("负beta拒绝",lambda:reinforce_loss(x,ref,reward,-0.1))
    c.raises("不一致组形状拒绝",lambda:reinforce_loss(x,ref[:,:2],reward,0.1))
    km=exact_kl_gradient_demo()
    c.ok("KL正则精确梯度与停止奖励估计一致",km['correct_max_error']<1e-12)
    c.ok("不detach塑形奖励产生错误梯度",km['wrong_max_error']>1e-3)
    sm=stale_sampling_demo()
    c.close("重要性权重在有限分布中校正期望",sm['correct_on_policy'],sm['exact_importance_weighted'])
    c.ok("旧数据无校正有偏",abs(sm['stale_unweighted'][0]-sm['correct_on_policy'][0])>0.1)

    base=TinyLM(ModelConfig()).eval()
    reference=freeze_reference(base)
    policy=inject_lora(copy.deepcopy(base)).eval()
    c.ok("参考独立存储",all(a.data_ptr()!=b.data_ptr() for a,b in zip(base.parameters(),reference.parameters())))
    c.ok("参考全部冻结",all(not p.requires_grad for p in reference.parameters()))
    c.ok("只训练adapter",all(p.requires_grad==is_adapter(n) for n,p in policy.named_parameters()))
    with torch.no_grad():
        c.close("初始策略等于参考",policy(batch['input_ids']),reference(batch['input_ids']),atol=0,rtol=0)
    # 只有两个生成步骤时，列出全部终止路径；概率总和应为1。
    all_answers=[[EOS]]+[[d,a] for d in codec.answer_ids[:-1] for a in codec.answer_ids]
    all_batch=collate_completions([rows[0]]*len(all_answers),all_answers,codec,24)
    with torch.no_grad():
        probs,_=completion_logps(base(all_batch['input_ids']),all_batch['labels'],codec)
    c.close("有限视野完整轨迹概率总和1",probs.exp().sum(),1.,atol=2e-6,rtol=2e-6)
    gen=torch.Generator().manual_seed(456)
    policy.train()
    roll=sample_rollouts(policy,rows,codec,4,gen)
    c.ok("采样后恢复模型模式",policy.training)
    c.ok("4个独立槽不做去重",len(roll['samples'])==8)
    c.ok("生成无计算图",all(type(s['behavior_logp']) is float for s in roll['samples']))
    c.ok("样本支持集合正确",all(t in codec.answer_ids for a in roll['answers'] for t in a))
    c.ok("EOS之后没有输出",all(EOS not in a[:-1] for a in roll['answers']))
    b2=collate_completions(roll['rows'],roll['answers'],codec,24)
    logs,_=completion_logps(policy(b2['input_ids']),b2['labels'],codec)
    c.close("逐步采样概率等于教师强制重算",logs.detach(),[s['behavior_logp'] for s in roll['samples']],atol=2e-5,rtol=2e-5)
    # 调整参考答案不会改变生成token；奖励端才读取答案。
    changed=copy.deepcopy(rows)
    for row in changed: row['answer']='99'
    r1=sample_rollouts(policy,rows,codec,2,torch.Generator().manual_seed(10))
    r2=sample_rollouts(policy,changed,codec,2,torch.Generator().manual_seed(10))
    c.ok("生成不读取参考答案",r1['answers']==r2['answers'])
    opt=torch.optim.Adam([p for p in policy.parameters() if p.requires_grad],lr=0.003)
    before_base=tensor_hash(base_state(policy)); before_ref=tensor_hash(reference.state_dict())
    bad=copy.deepcopy(roll); bad['samples'][0]['reward']=1-bad['samples'][0]['reward']
    c.raises("篡改奖励拒绝",lambda:one_on_policy_step(policy,reference,bad,codec,opt,0.2,'loo'))
    bad=copy.deepcopy(roll); bad['samples'][0]['behavior_logp']+=1
    c.raises("采样概率不匹配拒绝",lambda:one_on_policy_step(policy,reference,bad,codec,opt,0.2,'loo'))
    bad=copy.deepcopy(roll); bad['greedy']=True
    c.raises("贪心数据不是on-policy样本",lambda:one_on_policy_step(policy,reference,bad,codec,opt,0.2,'loo'))
    bad=copy.deepcopy(roll); bad['rows'][0]['split']='test'
    c.raises("测试题不进入训练",lambda:one_on_policy_step(policy,reference,bad,codec,opt,0.2,'loo'))
    bad=copy.deepcopy(roll); bad['policy_hash']='wrong'
    c.raises("拒绝陈旧策略样本",lambda:one_on_policy_step(policy,reference,bad,codec,opt,0.2,'loo'))
    bad=copy.deepcopy(roll); bad['samples'][0]['group_index']=1
    c.raises("拒绝组内错配",lambda:one_on_policy_step(policy,reference,bad,codec,opt,0.2,'loo'))
    stats=one_on_policy_step(policy,reference,roll,codec,opt,0.2,'loo')
    c.ok("更新后样本标为已使用",roll['used'])
    c.ok("更新不修改冻结基座",tensor_hash(base_state(policy))==before_base)
    c.ok("更新不修改参考",tensor_hash(reference.state_dict())==before_ref)
    c.ok("参考无残留梯度",all(p.grad is None for p in reference.parameters()))
    c.ok("所有adapter梯度有限",all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in policy.parameters() if p.requires_grad))
    c.ok("记录实际生成token数",stats['generated_tokens']==sum(len(a) for a in roll['answers']))
    c.raises("同一批样本禁止第二次更新",lambda:one_on_policy_step(policy,reference,roll,codec,opt,0.2,'loo'))
    # 用固定分数核对梯度方向，不把这个人为样本当作在线采样轨迹。
    check_lp=torch.tensor([[-1.,-1.]],dtype=torch.float64,requires_grad=True)
    check_r=torch.tensor([[1.,0.]],dtype=torch.float64)
    check_loss,_=reinforce_loss(check_lp,check_lp.detach(),check_r,0.,'loo')
    check_grad=torch.autograd.grad(check_loss,check_lp)[0]
    c.ok("正优势推高该样本log概率",check_grad[0,0]<0 and check_grad[0,1]>0)
    with tempfile.TemporaryDirectory() as tmp:
        path=Path(tmp)/'adapter.pt'
        torch.save(adapter_snapshot(policy),path)
        restored=inject_lora(copy.deepcopy(base)).eval()
        load_adapter(restored,torch.load(path,weights_only=True))
        with torch.no_grad():
            c.close("adapter磁盘重载输出相同",policy(batch['input_ids']),restored(batch['input_ids']),atol=0,rtol=0)
    saved_grad=[None if p.grad is None else p.grad.clone() for p in policy.parameters()]
    ev=evaluate_policy(policy,splits['dev'][:2],codec,2)
    c.ok("评测有固定分母",ev['tasks']==2 and ev['sampled_total']==4)
    c.ok("精确成功概率范围",0<=ev['exact_success_probability']<=1)
    c.ok("评测不修改已有梯度",all((a is None and p.grad is None) or (a is not None and torch.equal(a,p.grad)) for a,p in zip(saved_grad,policy.parameters())))
    c.raises("G=1配置拒绝",lambda:TrainConfig(group_size=1).validate())
    c.raises("非法最大步数拒绝",lambda:TrainConfig(max_new_tokens=2).validate())
    return {"passed":len(c.names),"names":c.names,"scope":"CPU数学与小型采样训练；不是LLM/GPU成绩"}

In [27]:
checks = run_checks()
print("固定检查通过：", checks["passed"], "项")
if RUN_DIR is not None:
    write_json(RUN_DIR/"checks.json", checks)

固定检查通过： 87 项


In [28]:
# %% 09 独立练习的检查入口

def exercise_checks_a(fn: Callable) -> dict:
    values=torch.tensor([[1.,0.,0.,0.],[3.,3.,3.,3.]],dtype=torch.float64,requires_grad=True)
    snapshot=values.detach().clone(); answer=fn(values)
    require(torch.allclose(answer,leave_one_out(values)),"LOO数值不正确")
    require(not answer.requires_grad,"应停止baseline/advantage梯度")
    require(torch.equal(values.detach(),snapshot),"不能原地修改输入")
    try: fn(torch.ones(1,1))
    except ValueError: pass
    else: raise LessonError("G=1应该拒绝")
    return {"exercise":"A","passed":True}


def exercise_checks_b(fn: Callable) -> dict:
    codec=CharCodec(); rows=make_splits()['train'][:2]
    batch=collate_completions(rows,[codec.encode(rows[0]['answer'])+[EOS],[EOS]],codec,24)
    logits=torch.randn(2,batch['input_ids'].shape[1],20,dtype=torch.float64,requires_grad=True)
    expected,_=completion_logps(logits,batch['labels'],codec)
    got=fn(logits,batch['labels'],codec)
    require(got.shape==(2,) and torch.allclose(got,expected,atol=1e-10,rtol=1e-10),"移位/动作集合/序列求和错误")
    g1=torch.autograd.grad(got.sum(),logits,retain_graph=True)[0]
    g2=torch.autograd.grad(expected.sum(),logits)[0]
    require(torch.allclose(g1,g2,atol=1e-10,rtol=1e-10),"log概率梯度错误")
    return {"exercise":"B","passed":True}


def exercise_checks_c(fn: Callable) -> dict:
    lp=torch.tensor([[-1.,-2.,-1.5],[-.2,-.4,-.3]],dtype=torch.float64,requires_grad=True)
    ref=torch.tensor([[-1.2,-1.6,-2.1],[-.3,-.3,-.3]],dtype=torch.float64,requires_grad=True)
    reward=torch.tensor([[1.,0.,0.],[0.,1.,1.]],dtype=torch.float64,requires_grad=True)
    expected,_=reinforce_loss(lp,ref,reward,0.2,'loo')
    got=fn(lp,ref,reward,0.2)
    require(got.ndim==0 and torch.allclose(got,expected,atol=1e-10,rtol=1e-10),"策略代理损失数值错误")
    g=torch.autograd.grad(got,(lp,ref,reward),retain_graph=True,allow_unused=True)
    true_g=torch.autograd.grad(expected,lp)[0]
    require(torch.allclose(g[0],true_g,atol=1e-10,rtol=1e-10),"策略梯度错误")
    require(g[1] is None and g[2] is None,"参考或奖励被错误反向")
    return {"exercise":"C","passed":True}

## 17. 独立练习 A：留一法优势

实现 `loo_ex(values)`，输入为 `[B,G]`。要求每一行只使用同一问题的其他样本作为 baseline，G至少为2，不原地修改数据，返回值停止梯度。

先手算 `[1,0,0,0]`、`[1,1,1,1]`，再写代码。

In [29]:
def loo_ex(values):
    raise NotImplementedError("请实现同题留一法优势")

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    print(exercise_checks_a(loo_ex))

## 18. 独立练习 B：回答序列 log 概率

实现 `sequence_logps_ex(logits, labels, codec)`，返回 `[B]`。

使用相同的回答动作集合；正确移位一次；排除prompt和padding，保留EOS；沿序列求和，保留当前策略的梯度。不要返回浮点 Python 数值或 `.item()` 的列表。

In [30]:
def sequence_logps_ex(logits, labels, codec):
    raise NotImplementedError("请实现回答log概率之和")

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    print(exercise_checks_b(sequence_logps_ex))

## 19. 独立练习 C：KL塑形＋LOO的代理损失

实现 `policy_loss_ex(logps, ref_logps, rewards, beta)`，输入均为 `[B,G]`，返回可反向的标量。

要求参考与塑形回报不接受梯度，当前 logps 接受梯度；不把 sampled log ratio 截成非负，不添加第二份重复KL项，不对优势做额外标准差归一化。

In [31]:
def policy_loss_ex(logps, ref_logps, rewards, beta):
    raise NotImplementedError("请实现KL塑形与LOO损失")

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    print(exercise_checks_c(policy_loss_ex))

## 20. 参考答案

先完成上面的独立实现，再查看。以下使用独立名称，不覆盖你的函数。它们突出核心运算；完整训练接口还包含前面展示的输入与身份校验。

In [32]:
def reference_a(values):
    if values.ndim!=2 or values.shape[1]<2:
        raise ValueError("需要[B,G]，G>=2")
    detached=values.detach()
    others=(detached.sum(1,keepdim=True)-detached)/(detached.shape[1]-1)
    return detached-others


def reference_b(logits,labels,codec):
    targets=labels[:,1:]
    valid=targets.ne(IGNORE)
    ids=targets.masked_fill(~valid,EOS)
    lookup=labels.new_full((len(codec.tokens),),-1)
    lookup[codec.answer_ids]=torch.arange(len(codec.answer_ids),device=labels.device)
    local=lookup[ids]
    logp=F.log_softmax(logits[:,:-1,codec.answer_ids],dim=-1)
    selected=logp.gather(-1,local[...,None]).squeeze(-1)
    return torch.where(valid,selected,torch.zeros_like(selected)).sum(1)


def reference_c(logps,ref_logps,rewards,beta):
    shaped=(rewards.detach()-beta*(logps.detach()-ref_logps.detach())).detach()
    others=(shaped.sum(1,keepdim=True)-shaped)/(shaped.shape[1]-1)
    advantages=shaped-others
    return -(advantages.detach()*logps).mean()

In [33]:
print(exercise_checks_a(reference_a))
print(exercise_checks_b(reference_b))
print(exercise_checks_c(reference_c))

{'exercise': 'A', 'passed': True}
{'exercise': 'B', 'passed': True}
{'exercise': 'C', 'passed': True}


## 21. 本课验收与容易混淆的边界

| 现象 | 应该怎样检查 |
|---|---|
| `reward.backward()` 不更新策略 | 奖励与生成参数是否有路径？应从 log 概率连接 |
| 一组全错、优势全零 | 在 beta=0 的LOO中是预期结果；检查探索与SFT起点 |
| sampled log ratio为负 | 不是精确全分布KL；不要未经推导直接clamp |
| 代理loss下降 | 还需检查真实任务概率、采样成功、贪心结果 |
| 同一批训练多轮 | 已不满足本课的一步on-policy约定，不能直接复用 |
| 温度/Top-p只改生成 | 重算概率是否还是相同分布？本课不启用这些变化 |
| 参考模型与当前模型同一对象 | 参考会移动；检查独立存储、冻结与内容哈希 |
| baseline无偏 | 仅指相应条件下原始梯度估计的期望，不保证有限训练轨迹最优 |

当前训练无需可微奖励，也没有额外奖励模型或价值网络。它是序列级 REINFORCE/RLOO 教学实现，**不是 PPO、不是完整 GRPO，也不是直接调用TR​L的结果**。

完成练习后，应能解释：为什么正优势对应负的 loss 梯度；为什么独立其他样本可作 baseline；为什么 KL 塑形需要正确的停止梯度；为什么一批只更新一次。

**下一课：第60课——重要性采样、概率比与PPO-Clip。** 区分行为旧策略和固定参考，研究怎样复用数据，以及裁剪目标改变了什么。

## 参考资料与本课实验范围

1. [Spinning Up：策略梯度与Baseline推导](https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html)
2. [Back to Basics：REINFORCE与留一法](https://arxiv.org/html/2402.14740v1)
3. [TRL RLOO：生成、回报、KL和优势](https://huggingface.co/docs/trl/rloo_trainer)
4. [PyTorch：分布、sample与log_prob](https://docs.pytorch.org/docs/2.10/distributions.html)
5. [PyTorch：multinomial](https://docs.pytorch.org/docs/2.10/generated/torch.multinomial.html)
6. [PyTorch：复现边界](https://docs.pytorch.org/docs/2.10/notes/randomness.html)

本课的全枚举、模型训练、评分与重载由随附代码实际计算。文献中的模型效果不是本课成绩。制作环境使用CPU；不报告3060的速度、显存或真实大模型效果。固定版本链接用于追溯接口，不宣称是最新版本。